In [1]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 156.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 59.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [9]:
!pip install -U llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 39.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.1 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.21-py3-none-linux_x86_64.whl size=18273645 sha256=0dfc9773590f12c353f03384ffe6c384f03354aebfcfa27018d0f113ee9916cb
  Stored in directory: /root/.cache/pip/wheels/8d/9c/44/39cb3a9e47ced67e64197053dace3cec12faf53daf81cac6c4
Successfully built llama-cpp-python


In [5]:
# Login to HuggingFace (required for gated MedGemma model)
# Make sure you have accepted HAI-DEF terms at:
# https://huggingface.co/google/medgemma-4b-it
from huggingface_hub import login
login()  # Paste your hf_... token when prompted

## Local Inference on GPU
Model page: https://huggingface.co/unsloth/medgemma-27b-text-it-GGUF

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/unsloth/medgemma-27b-text-it-GGUF)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [14]:
from llama_cpp import Llama
from huggingface_hub import hf_hub_download

repo_id = "unsloth/medgemma-27b-text-it-GGUF"
filename_1 = "BF16/medgemma-27b-text-it-BF16-00001-of-00002.gguf"
filename_2 = "BF16/medgemma-27b-text-it-BF16-00002-of-00002.gguf"

# Download both parts of the sharded model
# The from_pretrained method should handle this, but explicit download ensures both files are present.
model_path_1 = hf_hub_download(repo_id=repo_id, filename=filename_1)
model_path_2 = hf_hub_download(repo_id=repo_id, filename=filename_2)

# Initialize Llama with the path to the first file.
# llama_cpp will automatically look for other shards in the same directory.
llm = Llama(model_path=model_path_1)

BF16/medgemma-27b-text-it-BF16-00001-of-(…):   0%|          | 0.00/49.9G [00:00<?, ?B/s]

BF16/medgemma-27b-text-it-BF16-00002-of-(…):   0%|          | 0.00/4.13G [00:00<?, ?B/s]

llama_model_loader: additional 1 GGUFs metadata loaded.
llama_model_loader: loaded meta data with 47 key-value pairs and 808 tensors from /root/.cache/huggingface/hub/models--unsloth--medgemma-27b-text-it-GGUF/snapshots/334fbf6811c963d223f6ac107a459347353f068d/BF16/medgemma-27b-text-it-BF16-00001-of-00002.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Medgemma-27B-Text-It
llama_model_loader: - kv   3:                           general.finetune str              = text-it
llama_model_loader: - kv   4:                           general.basename str              = Medgemma-27B-Text-It
llama_model_loader: - kv   5:           

In [16]:
import time

def medgemma_analyze(llm, report_text: str, max_new_tokens: int = 400) -> dict:
    """
    Analyze medical report with MedGemma (using llama_cpp Llama object).
    Returns dict with analysis text and timing.
    """
    prompt = f"""Analyze this medical report and provide:
1. Key clinical findings (bullet points)
2. Medical conditions/diagnoses mentioned
3. Medications and dosages
4. Lab values — note if normal or abnormal
5. Clinical severity: Mild / Moderate / Severe / Critical

Medical Report:
---
{report_text}
---

Analysis:"""

    messages = [{
        'role': 'user',
        'content': prompt
    }]

    start = time.time()
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=0.0 # for consistent output
    )
    elapsed = time.time() - start

    # Extract the generated text
    result_text = response["choices"][0]["message"]["content"]
    tokens_generated = response["usage"]["completion_tokens"]

    return {
        'text': result_text.strip(),
        'time_seconds': round(elapsed, 1),
        'tokens_generated': tokens_generated
    }


def medgemma_qa(llm, report_text: str, question: str, max_new_tokens: int = 200) -> dict:
    """
    Answer a clinical question about a report using MedGemma (using llama_cpp Llama object).
    """
    prompt = f"""Based on this medical report, answer the question clinically.

Report:
{report_text}

Question: {question}

Clinical Answer:"""

    messages = [{
        'role': 'user',
        'content': prompt
    }]

    start = time.time()
    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=0.0 # for consistent output
    )
    elapsed = time.time() - start

    result_text = response["choices"][0]["message"]["content"]
    tokens_generated = response["usage"]["completion_tokens"]

    return {
        'text': result_text.strip(),
        'time_seconds': round(elapsed, 1),
        'tokens_generated': tokens_generated
    }


print('MedGemma inference functions ready.')

MedGemma inference functions ready.


In [17]:
# These are the SAME reports used in the VS Code app
# This ensures a fair apples-to-apples comparison

REPORTS = {
    'diabetes_followup': {
        'label': 'Diabetes Follow-up (Expected: Moderate urgency)',
        'text': """Patient: Alex Johnson, 52M
Visit: Diabetes follow-up
Vitals: BP 138/86, Weight 198 lbs (up 4 lbs)
Meds: Metformin 500mg BID, Lisinopril 10mg QD
Labs: HbA1c 8.2%, Fasting glucose 186 mg/dL, eGFR 62
Assessment: Type 2 DM suboptimal control. Early diabetic nephropathy. HTN not at goal.
Plan: Increase Metformin to 1000mg BID. Add Empagliflozin 10mg QD. Recheck in 3 months."""
    },
    'chest_xray': {
        'label': 'Chest X-Ray Report (Expected: High urgency)',
        'text': """Patient: Maria Rodriguez, 67F
Study: PA chest radiograph
Indication: Progressive dyspnea, productive cough x 10 days
Findings: Right lower lobe consolidation with air bronchograms.
Small right pleural effusion. Increased interstitial markings bilaterally.
Impression: Right lower lobe pneumonia. Small parapneumonic effusion.
Recommendation: Urgent clinical evaluation. Consider sputum culture.
Follow-up imaging in 4-6 weeks."""
    },
    'routine_checkup': {
        'label': 'Routine Annual Checkup (Expected: Low urgency)',
        'text': """Patient: David Chen, 34M
Visit: Annual preventive care
Vitals: BP 118/74, BMI 23.2
Labs: Total cholesterol 182, LDL 98, HDL 58, Glucose 92, TSH 2.1. CBC normal.
Assessment: Healthy male. All screening labs within normal limits.
Plan: Continue current lifestyle. Annual flu vaccine given. Return in 1 year."""
    }
}

print(f'Loaded {len(REPORTS)} sample reports:')
for key, r in REPORTS.items():
    print(f'  - {key}: {r["label"]}')

Loaded 3 sample reports:
  - diabetes_followup: Diabetes Follow-up (Expected: Moderate urgency)
  - chest_xray: Chest X-Ray Report (Expected: High urgency)
  - routine_checkup: Routine Annual Checkup (Expected: Low urgency)


In [18]:
# Run MedGemma on all 3 reports and collect results
medgemma_results = {}

for report_key, report_data in REPORTS.items():
    print(f'\n{'='*60}')
    print(f'Report: {report_data["label"]}')
    print(f'{'='*60}')
    print('Sending to MedGemma...')

    result = medgemma_analyze(llm, report_data['text'])
    medgemma_results[report_key] = result

    print(f'Done in {result["time_seconds"]}s | {result["tokens_generated"]} tokens generated')
    print(f'\nMedGemma Output:')
    print(result['text'])


Report: Diabetes Follow-up (Expected: Moderate urgency)
Sending to MedGemma...


llama_perf_context_print:        load time =   40753.90 ms
llama_perf_context_print: prompt eval time =   40753.46 ms /   223 tokens (  182.75 ms per token,     5.47 tokens per second)
llama_perf_context_print:        eval time =  364426.16 ms /   288 runs   ( 1265.37 ms per token,     0.79 tokens per second)
llama_perf_context_print:       total time =  405619.77 ms /   511 tokens
llama_perf_context_print:    graphs reused =        286
Llama.generate: 69 prefix-match hit, remaining 102 prompt tokens to eval


Done in 405.6s | 289 tokens generated

MedGemma Output:
Here's an analysis of the provided medical report:

**1. Key Clinical Findings:**

*   Patient is a 52-year-old male.
*   Follow-up visit for Diabetes Mellitus (DM).
*   Blood pressure (BP) is elevated at 138/86 mmHg.
*   Weight has increased by 4 lbs since the last visit.
*   HbA1c is 8.2%, indicating poor glycemic control over the past 2-3 months.
*   Fasting glucose is elevated at 186 mg/dL.
*   Estimated Glomerular Filtration Rate (eGFR) is 62 mL/min/1.73m², suggesting mild kidney function impairment.

**2. Medical Conditions/Diagnoses Mentioned:**

*   Type 2 Diabetes Mellitus (DM) - suboptimal control
*   Hypertension (HTN) - not at goal
*   Early Diabetic Nephropathy (kidney disease related to diabetes)

**3. Medications and Dosages:**

*   **Current:**
    *   Metformin 500mg twice daily (BID)
    *   Lisinopril 10mg once daily (QD)
*   **Planned Changes:**
    *   Metformin increased to 1000mg twice daily (BID)

Report: C

llama_perf_context_print:        load time =   40753.90 ms
llama_perf_context_print: prompt eval time =   19501.89 ms /   102 tokens (  191.19 ms per token,     5.23 tokens per second)
llama_perf_context_print:        eval time =  429464.96 ms /   340 runs   ( 1263.13 ms per token,     0.79 tokens per second)
llama_perf_context_print:       total time =  449513.43 ms /   442 tokens
llama_perf_context_print:    graphs reused =        338
Llama.generate: 69 prefix-match hit, remaining 108 prompt tokens to eval


Done in 449.5s | 341 tokens generated

MedGemma Output:
Okay, here's an analysis of the provided medical report for Maria Rodriguez:

**1. Key Clinical Findings:**

*   **Progressive dyspnea:** Worsening shortness of breath.
*   **Productive cough:** Cough producing sputum.
*   **Duration:** Symptoms present for 10 days.
*   **Radiographic Findings:**
    *   Right lower lobe consolidation (area of lung tissue filled with fluid/pus).
    *   Air bronchograms within the consolidation (air-filled bronchi visible against the opaque lung tissue).
    *   Small right pleural effusion (fluid accumulation in the space between the lung and the chest wall on the right side).
    *   Increased interstitial markings bilaterally (suggestive of inflammation or fluid in the lung tissue framework on both sides).

**2. Medical Conditions/Diagnoses Mentioned:**

*   **Right lower lobe pneumonia:** Infection causing inflammation and consolidation in the lower part of the right lung.
*   **Small parapneu

llama_perf_context_print:        load time =   40753.90 ms
llama_perf_context_print: prompt eval time =   20787.13 ms /   108 tokens (  192.47 ms per token,     5.20 tokens per second)
llama_perf_context_print:        eval time =  422963.52 ms /   334 runs   ( 1266.36 ms per token,     0.79 tokens per second)
llama_perf_context_print:       total time =  444281.75 ms /   442 tokens
llama_perf_context_print:    graphs reused =        332


Done in 444.3s | 335 tokens generated

MedGemma Output:
Here's an analysis of the provided medical report:

**1. Key Clinical Findings:**

*   Patient is a 34-year-old male.
*   Visit was for annual preventive care.
*   Blood pressure (BP) is 118/74 mmHg.
*   Body Mass Index (BMI) is 23.2 kg/m².
*   All screening labs are within normal limits.
*   Patient received an annual flu vaccine.

**2. Medical Conditions/Diagnoses Mentioned:**

*   Healthy male (This is the overall assessment, not a specific diagnosis).
*   No specific medical conditions or diagnoses were identified during this visit.

**3. Medications and Dosages:**

*   No medications are mentioned in the report. The patient is not currently prescribed any medications based on this report.
*   Flu vaccine administered (dosage not specified, as it's a vaccine).

**4. Lab Values:**

*   **Total Cholesterol:** 182 mg/dL (Normal)
*   **LDL Cholesterol:** 98 mg/dL (Normal)
*   **HDL Cholesterol:** 58 mg/dL (Normal)
*   **Glucose:**